In [21]:
import os
import numpy as np
from scipy import stats
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [22]:
from factor_analyzer import FactorAnalyzer

def compute_cct(dataframe, n_factors = 3, return_ev = False):
    
    '''
    Function returns the cultural consensus theory answer for a given dataset [emotion tracking dataset]
    
    Parameters
    ----------
    dataframe: pandas dataframe where each column is an individual observers response
    n_factors: number of factors to fit to dataframe
    return_ev: when set to True, returns the eigenvalue ratio between the first and second factor
    
    Notes
    -----
    When the eigenvalue ratio is >= 3, we can assume that their is a correct answer [1]. This suggests
    that factor 1 explains 3 times the variance in the data than factor 2. However, we assume that
    there is a correct answer for any given video so we do not look into the eigenvalue ratio right now.
    
    We set number of factors as 3 as default. There should only be one correct answer but the answer
    could be bi-modal resulting in 2 factors. Thus, three factors (three correct answers) is a 
    conservative number.
    
    References
    ----------
    [1] https://journals.sagepub.com/doi/10.1177/1525822X07303502
    
    
    Returns
    -------
    default:returns the cct consensus
    if return_ev == True: returns the cct consensus and the eigenvalue ratio between factor 1 and factor 2
        
    '''
    
    # create factor analysis
    fa = FactorAnalyzer(rotation = None)
    fa.fit(dataframe, n_factors) # fit to data, with n number of factors
    
    # Check Eigenvalues
    ev, v = fa.get_eigenvalues() # get eigenvales (measure of explained variance by each factor)
    ev_ratio = ev[0]/ev[1] # eigenvalue ratio between factor 1 and 2, should be >= 3.
    
    # CCT mean
    cct_m = fa.transform(dataframe)[:,0] # get weighted average of the dataset using factor 1
    
    # check if returning eigenvalue ratio
    if return_ev:
        return cct_m, ev_ratio 
    
    else:
        return cct_m 
    
    
def compute_ewe(dataframe):

    w = []
    s = dataframe.columns

    # get correlation of each subject to average
    for s in dataframe.columns:

        r1 = dataframe.drop(columns = [s]).mean(axis = 1)
        r,p = stats.pearsonr(r1,dataframe[s])

        w.append(r)

    ws = dataframe*np.array(w)
    ws = ws.mean(axis = 1)*1/sum(np.array(w))
    
    return(ws)
    
def normalize_corr(signal, a = -1, b = 1):
    
    A = signal.min().min()
    B = signal.max().max()
    C = (a-b)/(A-B)
    k = (C*A - a)/C
    return (signal-k)*C

def converging_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[(n+j)*2 for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

def diverging_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[-(n+j)*2 for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

def neighbor_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[-abs(j-n) for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

# code to remove blinks and artifacts from the data

def outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xz[xz > z] = np.nan # remove values above 3 z score
    xz[xz < -z] = np.nan # remove values below -3 z score
    xz = xz.fillna(method='ffill') # change nans to previous value
    xz = xz.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xz

def emp_outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xc[xz > z] = np.nan # remove values above 3 z score
    xc[xz < -z] = np.nan # remove values below -3 z score
    xc = xc.fillna(method='ffill') # change nans to previous value
    xc = xc.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xc

def corrmean(data):
    return np.tanh(np.nanmean(np.arctanh(data)))

def bootstrap_mean_corr(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.tanh(np.nanmean(np.arctanh(x))) for x in boot_data] # get mean of each dataset
    
    m = np.tanh(np.nanmean(np.arctanh(means))) # fisher z transform
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

def bootstrap_mean(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.nanmean(x) for x in boot_data] # get mean of each dataset
    
    m = np.nanmean(means)  
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

In [23]:
questionnaireDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/exp2_questionnaire_preprocessed.csv'
surveyDF = pd.read_csv(questionnaireDir)
surveyDF

,PID,Gender,Age,Born US,Education,EQ,AQ
0,BF083,Female,18,Yes,"High school graduate, diploma or the equivalen...",121,16
1,BF083,Female,18,Yes,"High school graduate, diploma or the equivalen...",125,17
2,BF056,Male,18,Yes,"High school graduate, diploma or the equivalen...",147,18
3,BF102,Male,18,Yes,"High school graduate, diploma or the equivalen...",117,13
4,BF026,Male,18,Yes,"High school graduate, diploma or the equivalen...",125,11
...,...,...,...,...,...,...,...
100,BF016,Female,32,No,"Some college credit, no degree",98,32
101,BF070,Female,32,No,"Master's degree (for example: MA, MS, MEng, ME...",136,21
102,BF071,Male,37,Yes,"Some college credit, no degree",155,26
103,BF010,Non-binary/non-conforming,37,Yes,"Associate degree (for example: AA, AS)",143,10


In [25]:
### calculate average deviation of fixation

fixation = {} # store by subjects

dataDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/experiment 2/'
os.chdir(dataDir)

# iterate through folders
for folder in os.listdir():

    os.chdir(dataDir + folder)
    # check if path is a folder
    if not os.path.isdir(dataDir + folder):
        continue # skip if not a folder
        
    # update directory with folder
    os.chdir(dataDir + folder)
    
    # read the gaze csv
    df = pd.read_csv(folder + '_drift.csv').iloc[:,1:]
    
    # get x and y gaze coordinates 
    xcoord = [x for x in list(df.columns) if 'driftX' in x]
    xcoord = df[xcoord].dropna()#[:54]
    
    
    # y
    ycoord = [x for x in list(df.columns) if 'driftY' in x]
    ycoord = df[ycoord].dropna()#[:54]
    
    # store data
    fixation[folder] = {'x':xcoord, 'y': ycoord}
    
# reset dir
os.chdir(dataDir)

In [26]:
subj_xerror = {x:[] for x in fixation.keys()}
subj_yerror = {x:[] for x in fixation.keys()}

for subj in fixation.keys():
    x = emp_outlier_remover(fixation[subj]['x'].dropna())
    y = emp_outlier_remover(fixation[subj]['x'].dropna())

    x_centered = x - np.mean(x)
    y_centered = y - np.mean(y)
    
    xerror = x_centered.std().mean()
    yerror = y_centered.std().mean()
       
    subj_xerror[subj] = xerror
    subj_yerror[subj] = yerror

# sort list of subjects
avg_error = {s:(subj_xerror[s] + subj_yerror[s])/2 for s in subj_xerror.keys()}
sorted_avg = sorted(avg_error.items(), key=lambda x: x[1])
sorted_avg

[('BF070_1', 0.009959349815165379),
 ('BF036_1', 0.015326010872654486),
 ('BF021_1', 0.015847726338216843),
 ('BF082_1', 0.01805208593346917),
 ('BF022_1', 0.023359268988561818),
 ('BF078_2', 0.02737690772979231),
 ('BF055_2', 0.02916424403669536),
 ('BF062_1', 0.031194934703534058),
 ('BF104_1', 0.032180302231784974),
 ('BF075_1', 0.034270125366187176),
 ('BF100_1', 0.038546705043371196),
 ('BF016_1', 0.03892988183085208),
 ('BF069_1', 0.04114444692593218),
 ('BF049_1', 0.041237729444411717),
 ('BF046_1', 0.04348954766554822),
 ('BF083_2', 0.04480559602215621),
 ('BF053_1', 0.04483901961669388),
 ('BF071_1', 0.04740119680453116),
 ('BF057_1', 0.04743343200584589),
 ('BF019_1', 0.0504025580204367),
 ('BF066_2', 0.05239928307575424),
 ('BF025_1', 0.052756292754820786),
 ('BF064_1', 0.053034808637535465),
 ('BF013_1', 0.053250753924891966),
 ('BF041_1', 0.056924311385623816),
 ('BF073_1', 0.058154452748234504),
 ('BF040_1', 0.06308153778550156),
 ('BF014_1', 0.06621562332862335),
 ('BF06

In [27]:
# calculate error in visual angle

opp = 6.032/2 # distance of a .2 line in psychopy units
adj = 53.34   # distance from screen

va = math.degrees(math.atan(opp/adj))/2
dev = np.array(list(avg_error.values()))/.1
err = dev*va

subj_avgerror = {s:e for s,e in zip(list(avg_error.keys()),err)}

goodsubjects = [s for s in subj_avgerror.keys() if abs(subj_avgerror[s]) < 3]

# update questionnaire
surveyDF = surveyDF[surveyDF['PID'].isin([i[:-2] for i in goodsubjects])]

In [28]:
dataDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/experiment 2/'
os.chdir(dataDir)

context_rating_data = {}
context_gaze_data = {}

# iterate through folders
## gaze data
for folder in os.listdir():
    if folder not in goodsubjects:
        continue
    
    os.chdir(dataDir + folder)

    # identify and store core files
    gazeFile = [x for x in os.listdir() if 'gaze' in x][0]
    ratingFile = folder + '.csv'

    ## gaze data
    df = outlier_remover(pd.read_csv(gazeFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
            
        # store data

        if column not in context_gaze_data:
            context_gaze_data[column] = pd.DataFrame()
        context_gaze_data[column] = pd.concat([context_gaze_data[column],df[column].rename(folder)], axis = 1)

    ## rating data
    df = outlier_remover(pd.read_csv(ratingFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()

        # store data
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()

        context_rating_data[column] = pd.concat([context_rating_data[column],df[column].rename(folder)], axis = 1)
        
    os.chdir(dataDir)


In [29]:

## context condition
context_taskPerf = {}
context_taskPerf_valence = {}
context_taskPerf_arousal = {}

# iterate through the videos
for video in context_rating_data:
    if 'ground' in video:
        continue
    
    df = context_rating_data[video].dropna() # drop na rows

    for subj in df.columns:
        if subj not in context_taskPerf: # add subject to our dict
            context_taskPerf[subj] = []
            context_taskPerf_valence[subj] = []
            context_taskPerf_arousal[subj] = []

        sr = df[subj]
        cr = df.drop(columns = [subj]).mean(axis = 1) # traditional consensus
        # cr = compute_ewe(df.drop(columns = [subj])) # ewe consensus
        # cr = compute_cct(df.drop(columns = [subj]))
        
        r, p = stats.spearmanr(sr,cr)
        context_taskPerf[subj].append(r)

        if 'valence' in video:
            context_taskPerf_valence[subj].append(r)
        else:
            context_taskPerf_arousal[subj].append(r)

# compute average performance
context_avg_taskPerf = {s:corrmean(context_taskPerf[s]) for s in context_taskPerf}
context_avg_taskPerf_valence = {s:corrmean(context_taskPerf_valence[s]) for s in context_taskPerf_valence}
context_avg_taskPerf_arousal = {s:corrmean(context_taskPerf_arousal[s]) for s in context_taskPerf_arousal}


In [30]:
context_isc = {}

for video in context_gaze_data:
    if 'pupil' in video: # skip pupil data
        continue

    df = outlier_remover(context_gaze_data[video].dropna()) # remove blinks
    
    for subj in df.columns:
        if subj not in context_isc:
            context_isc[subj] = []

        sg = df[subj]
        cg = df.drop(columns = [subj]).mean(axis = 1)
        #cg = compute_cct(df.drop(columns = [subj])) # cct
        r, p = stats.pearsonr(sg,cg)
        context_isc[subj].append(r)   

# compute average ISC
avg_context_isc = {s:corrmean(context_isc[s]) for s in context_isc}


# bubbles

In [ ]:
x_df = emp_outlier_remover(context_gaze_data['011_gazeX'].dropna())
y_df = emp_outlier_remover(context_gaze_data['011_gazeY'].dropna())

In [73]:
from scipy.signal import resample

INPUT_VID = '/home/Jeff/1.0 projects/intersubject gaze correlations/videos/Exp2_011_context-only.mp4'
OUTPUT_VID = "/home/Jeff/1.0 projects/intersubject gaze correlations/videos/gaze_overlay_output.mp4"

x_df = context_gaze_data['011_gazeX']
y_df = context_gaze_data['011_gazeY']

# --- 1. Load Video and Get Properties ---
cap = cv2.VideoCapture(INPUT_VID)

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Input video: {frame_width}x{frame_height} @ {fps:.2f} FPS, {frame_count} frames total.")

participants = x_df.columns

x_resampled_data = {}
y_resampled_data = {}

for p_id in participants:
    # Get the data for this participant
    x_p = emp_outlier_remover(x_df[p_id].dropna())
    y_p = emp_outlier_remover( y_df[p_id].dropna())

    # Apply scipy.signal.resample
    # This takes the 1D array and the target number of samples
    x_resampled_array = resample(x_p, frame_count)
    y_resampled_array = resample(y_p, frame_count)
    
    # Store the resampled arrays
    x_resampled_data[p_id] = x_resampled_array
    y_resampled_data[p_id] = y_resampled_array

# Convert the dictionary of arrays into the final DataFrames
x_resampled = pd.DataFrame(x_resampled_data)
y_resampled = pd.DataFrame(y_resampled_data)


# --- 2. Setup Video Writer ---
print("Setting up video writer...")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VID, fourcc, fps, (frame_width, frame_height))

# --- 3. Define Plotting Parameters ---
radius = 20
alpha = 0.4
show_legend = False

# --- 4. Setup Participant Colors ---
colors = {}
for i, p_id in enumerate(participants):
    hue = int(i * (180 / len(participants))) 
    # BGR format
    colors[p_id] = tuple(int(c) for c in cv2.cvtColor(np.uint8([[[hue, 255, 255]]]), cv2.COLOR_HSV2BGR)[0][0])
# ================================================================
# --- 5. Process Video Frame by Frame (MODIFIED SECTION) ---
# ================================================================
print("Processing video frame by frame with coordinate scaling...")

# --- NEW: Define your gaze coordinate bounds ---
GAZE_X_MIN = -1.125 / 2
GAZE_X_RANGE = 1.125  # (1.125/2 - (-1.125/2))
GAZE_Y_MIN = -0.9 / 2
GAZE_Y_RANGE = 0.9     # (0.9/2 - (-0.9/2))
# ----------------------------------------------

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    overlay = frame.copy()

    try:
        current_x = x_resampled.iloc[frame_idx]
        current_y = y_resampled.iloc[frame_idx]
    except IndexError:
        print("Warning: Reached end of gaze data before video finished.")
        break

    for p_id in participants:
        # Get the raw gaze coordinate (e.g., -0.5625 to 0.5625)
        x_gaze = current_x[p_id] 
        y_gaze = current_y[p_id]

        if not pd.isna(x_gaze) and not pd.isna(y_gaze):
            
            # --- NEW: Scale gaze data to pixel coordinates ---
            
            # 1. Normalize gaze X from [GAZE_X_MIN, GAZE_X_MAX] to [0, 1]
            x_norm = (x_gaze - GAZE_X_MIN) / GAZE_X_RANGE
            
            # 2. Scale normalized X to pixel width [0, frame_width]
            x_pixel = int(x_norm * frame_width)
            
            # 1. Normalize gaze Y from [GAZE_Y_MIN, GAZE_Y_MAX] to [0, 1]
            #    (Here, 0 is the bottom, 1 is the top)
            y_norm = (y_gaze - GAZE_Y_MIN) / GAZE_Y_RANGE
            
            # 2. Scale normalized Y to pixel height [0, frame_height]
            #    AND INVERT it, because in OpenCV, y=0 is the TOP.
            y_pixel = int(frame_height * (1.0 - y_norm))
            
            # Create the final (x, y) pixel coordinate
            center_point = (x_pixel, y_pixel)
            # --- END NEW SCALING ---

            color = colors[p_id]
            # Draw the circle at the scaled pixel coordinate
            cv2.circle(overlay, center_point, radius, color, thickness=-1)

    # Blend the overlay with the original frame
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    
    # Add legend
    if show_legend:
        for i, p_id in enumerate(participants):
            color = colors[p_id]
            legend_text = f"P: {p_id}"
            pos = (15, 30 + i * 25)
            cv2.putText(frame, legend_text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 3, cv2.LINE_AA)
            cv2.putText(frame, legend_text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

    out.write(frame)
    frame_idx += 1

    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}/{frame_count}")

# --- 6. Release Resources ---
cap.release()
out.release()
print(f"\nSuccessfully created video: {OUTPUT_VID}")


Input video: 1280x720 @ 25.00 FPS, 2064 frames total.
Setting up video writer...
Processing video frame by frame with coordinate scaling...
Processed frame 100/2064
Processed frame 200/2064
Processed frame 300/2064
Processed frame 400/2064
Processed frame 500/2064
Processed frame 600/2064
Processed frame 700/2064
Processed frame 800/2064
Processed frame 900/2064
Processed frame 1000/2064
Processed frame 1100/2064
Processed frame 1200/2064
Processed frame 1300/2064
Processed frame 1400/2064
Processed frame 1500/2064
Processed frame 1600/2064
Processed frame 1700/2064
Processed frame 1800/2064
Processed frame 1900/2064
Processed frame 2000/2064

Successfully created video: /home/Jeff/1.0 projects/intersubject gaze correlations/videos/gaze_overlay_output.mp4


In [74]:
import cv2
import pandas as pd
import numpy as np
from scipy.signal import resample
import os
import glob     # New import for finding files
import imageio.v3 # New import for creating GIFs

# --- Assume you have these defined ---
# (You would load your data here)
# context_gaze_data = ... 
# def emp_outlier_remover(data):
#     # Your outlier removal function
#     return data
# -------------------------------------

# --- Using dummy data for a runnable example ---
# (Replace this with your actual data loading)
print("Creating dummy data for example...")
x_df = pd.DataFrame({
    'P1': (np.random.rand(1000) - 0.5) * 1.125, 
    'P2': (np.random.rand(1000) - 0.5) * 1.125
})
y_df = pd.DataFrame({
    'P1': (np.random.rand(1000) - 0.5) * 0.9, 
    'P2': (np.random.rand(1000) - 0.5) * 0.9
})
x_df.iloc[100:150, 0] = np.nan
y_df.iloc[300:350, 1] = np.nan
def emp_outlier_remover(data): return data
# -----------------------------------------------


# --- Your File Paths ---
INPUT_VID = '/home/Jeff/1.0 projects/intersubject gaze correlations/videos/Exp2_011_context-only.mp4'
OUTPUT_VID = "/home/Jeff/1.0 projects/intersubject gaze correlations/videos/gaze_overlay_output.mp4"

# --- NEW: GIF CREATION SETTINGS ---
SAVE_FRAMES_FOR_GIF = True  # Set to False to skip GIF creation
FRAME_SAVE_FOLDER = "/home/Jeff/1.0 projects/intersubject gaze correlations/videos/temp_frames"
GIF_OUTPUT_PATH = "/home/Jeff/1.0 projects/intersubject gaze correlations/videos/gaze_overlay.gif"

# Create the folder to store frames
if SAVE_FRAMES_FOR_GIF:
    os.makedirs(FRAME_SAVE_FOLDER, exist_ok=True)
    print(f"Saving individual frames to: {FRAME_SAVE_FOLDER}")
# ----------------------------------


# --- 1. Load Video and Get Properties ---
if not os.path.exists(INPUT_VID):
    print(f"Error: Video file not found at {INPUT_VID}")
    # Create a dummy video file just so the script can run
    print("Creating a dummy video file for testing...")
    cv2.VideoWriter('dummy_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (1920, 1080)).release()
    INPUT_VID = 'dummy_video.mp4' 

cap = cv2.VideoCapture(INPUT_VID)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {INPUT_VID}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if frame_count == 0:
    print("Warning: Video has 0 frames. Setting to 150 for dummy data.")
    frame_count = 150
    if fps == 0: fps = 30
    if frame_width == 0: frame_width = 1920
    if frame_height == 0: frame_height = 1080

print(f"Input video: {frame_width}x{frame_height} @ {fps:.2f} FPS, {frame_count} frames total.")
print(f"Original gaze data has {len(x_df)} samples.")

# --- Resampling Logic (unchanged) ---
participants = x_df.columns
x_resampled_data = {}
y_resampled_data = {}
print("Resampling gaze data...")
for p_id in participants:
    x_p, y_p = x_df[p_id], y_df[p_id]
    x_p_filled = x_p.interpolate(method='linear', limit_direction='both').ffill().bfill()
    y_p_filled = y_p.interpolate(method='linear', limit_direction='both').ffill().bfill()
    if x_p_filled.isnull().all(): x_p_filled = x_p_filled.fillna(0)
    if y_p_filled.isnull().all(): y_p_filled = y_p_filled.fillna(0)
    x_p_cleaned, y_p_cleaned = emp_outlier_remover(x_p_filled), emp_outlier_remover(y_p_filled)
    x_resampled_data[p_id] = resample(x_p_cleaned, frame_count)
    y_resampled_data[p_id] = resample(y_p_cleaned, frame_count)
x_resampled = pd.DataFrame(x_resampled_data)
y_resampled = pd.DataFrame(y_resampled_data)
print(f"Resampled data to {len(x_resampled)} frames.")

# --- 2. Setup Video Writer (unchanged) ---
print("Setting up video writer...")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VID, fourcc, fps, (frame_width, frame_height))

# --- 3. Plotting Parameters (unchanged) ---
radius, alpha, show_legend = 20, 0.4, True

# --- 4. Colors (unchanged) ---
colors = {}
for i, p_id in enumerate(participants):
    hue = int(i * (180 / len(participants))) 
    colors[p_id] = tuple(int(c) for c in cv2.cvtColor(np.uint8([[[hue, 255, 255]]]), cv2.COLOR_HSV2BGR)[0][0])

# --- 5. Coordinate Scaling (unchanged) ---
GAZE_X_MIN, GAZE_X_RANGE = -1.125 / 2, 1.125
GAZE_Y_MIN, GAZE_Y_RANGE = -0.9 / 2, 0.9

# --- 6. Process Video Frame by Frame ---
print("Processing video frame by frame...")
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    overlay = frame.copy()
    try:
        current_x = x_resampled.iloc[frame_idx]
        current_y = y_resampled.iloc[frame_idx]
    except IndexError:
        break

    for p_id in participants:
        x_gaze, y_gaze = current_x[p_id], current_y[p_id]
        if not pd.isna(x_gaze) and not pd.isna(y_gaze):
            x_norm = (x_gaze - GAZE_X_MIN) / GAZE_X_RANGE
            y_norm = (y_gaze - GAZE_Y_MIN) / GAZE_Y_RANGE
            x_pixel = int(x_norm * frame_width)
            y_pixel = int(frame_height * (1.0 - y_norm))
            center_point = (x_pixel, y_pixel)
            cv2.circle(overlay, center_point, radius, color, thickness=-1)

    # --- THIS IS THE FIX FROM LAST TIME ---
    # Blend 'overlay' and 'frame', saving the result back into 'frame'
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame) 
    
    if show_legend:
        for i, p_id in enumerate(participants):
            color = colors[p_id]
            legend_text = f"P: {p_id}"
            pos = (15, 30 + i * 25)
            cv2.putText(frame, legend_text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 3, cv2.LINE_AA)
            cv2.putText(frame, legend_text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

    # --- Write to MP4 (unchanged) ---
    out.write(frame)

    # --- NEW: SAVE FRAME TO FOLDER ---
    if SAVE_FRAMES_FOR_GIF:
        # Use zfill for zero-padding (e.g., 00001, 00002) so files sort correctly
        filename = f"frame_{frame_idx:05d}.png"
        save_path = os.path.join(FRAME_SAVE_FOLDER, filename)
        cv2.imwrite(save_path, frame)
    # ---------------------------------
        
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}/{frame_count}")

# --- 7. Release Video Resources ---
cap.release()
out.release()
print(f"\nSuccessfully created video: {OUTPUT_VID}")

# Clean up dummy video
if INPUT_VID == 'dummy_video.mp4':
    os.remove(INPUT_VID)
    print("Cleaned up dummy video file.")

# ======================================================
# --- 8. NEW: CREATE GIF FROM SAVED FRAMES ---
# ======================================================
if SAVE_FRAMES_FOR_GIF:
    print("\nCreating GIF from saved frames...")
    
    # Find all saved .png frames, sorting them by name
    filenames = sorted(glob.glob(os.path.join(FRAME_SAVE_FOLDER, "frame_*.png")))
    
    if not filenames:
        print(f"Error: No frames found in {FRAME_SAVE_FOLDER}. Skipping GIF creation.")
    else:
        print(f"Found {len(filenames)} frames. Reading images...")
        
        # Read all images into a list
        images = []
        for i, filename in enumerate(filenames):
            images.append(imageio.v3.imread(filename))
            if (i + 1) % 100 == 0:
                print(f"Read {i+1} images...")
        
        # Calculate frame duration from FPS
        # imageio duration is in seconds per frame
        frame_duration = 1.0 / fps
        
        print(f"Saving GIF to {GIF_OUTPUT_PATH} (this may take a moment)...")
        # Save as a GIF
        imageio.v3.imwrite(
            GIF_OUTPUT_PATH, 
            images, 
            duration=frame_duration, 
            loop=0  # 0 = loop forever
        )
        print("Successfully created GIF!")

        # --- Optional: Clean up frames ---
        # print(f"Cleaning up {len(filenames)} individual frames...")
        # for f in filenames:
        #     os.remove(f)
        # os.rmdir(FRAME_SAVE_FOLDER)
        # print("Cleanup complete.")

Creating dummy data for example...
Saving individual frames to: /home/Jeff/1.0 projects/intersubject gaze correlations/videos/temp_frames
Input video: 1280x720 @ 25.00 FPS, 2064 frames total.
Original gaze data has 1000 samples.
Resampling gaze data...
Resampled data to 2064 frames.
Setting up video writer...
Processing video frame by frame...
Processed frame 100/2064
Processed frame 200/2064
Processed frame 300/2064
Processed frame 400/2064
Processed frame 500/2064
Processed frame 600/2064
Processed frame 700/2064
Processed frame 800/2064
Processed frame 900/2064
Processed frame 1000/2064
Processed frame 1100/2064
Processed frame 1200/2064
Processed frame 1300/2064
Processed frame 1400/2064
Processed frame 1500/2064
Processed frame 1600/2064
Processed frame 1700/2064
Processed frame 1800/2064
Processed frame 1900/2064
Processed frame 2000/2064

Successfully created video: /home/Jeff/1.0 projects/intersubject gaze correlations/videos/gaze_overlay_output.mp4

Creating GIF from saved fra